# Regressione lineare multipla

Nella regressione lineare multipla l'obiettivo è stimare un output $Y$ a partire da più variabili indipendenti in input (**predittori**), prendendo un esempio reale supponiamo di voler preddirre le calorie bruciate a partire da:
- Tipo di allenamento.
- Peso.
- Altezza.
- Età.
- Intensità.
- Durata.

Siccome utilizzeremo più variabili in input allora la formula avrà tanti parametri di pendenza quanto il numero di variabili:
$$
y = \beta_0 + \beta_1 * \text{durata} + \beta_2 * \text{intensità} \dots
$$

Generalizzando la formula della regressione multipla può essere riscritta in forma matriciale come:

$$
y = X\beta + \epsilon
$$

dove:
- y sarà un vettore $n \times 1$.
- $X$ sarà una matrice $n \times (p+1)$, dove facciamo +1 per includere anche l'intercetta (la quale sarà impostata a 1).
- $\beta$ sarà un vettore $(p+1)\times 1$.

## Stima dei parametri

Il procedimento per la stima dei parametri segue le medesime formule viste in precedenza, ovvero usando il **metodo dei minimi quadrati (OLS)**, solo che in questo caso le formule verranno riportate in forma matriciale:

$$
\hat \beta = (X^T X)^{-1}X^{T} y \qquad S_e = \sqrt {\frac {\text{SSR}} {n-p-1}}
$$

Dove $\text{SSR}=||y-X\hat \beta||^2$.

## Selezione delle variabili

In tal caso potremmo utilizzare delle procedure di selezione delle variabili, come metodi **stepwise**, i più comuni sono:
- **Selezione backward**: si parte dal modello completo, quindi con tutte le variabili, e si eliminano una alla volta, quindi si partirà dal modello più complesso per semplificarlo progressivamente, la procedura prevedede di:
    1. **Inizio**: si stima il modello completo includendo tutti i predittori.
    2. **Verifica**: si calcolano i p-value per i test $t$ su ogni singolo coefficiente $\beta_j$ (con $H_0: \beta_j=0$).
    3. **Decisione**: 
        - Se tutti i _p-value_ sono inferiori a una soglia di significatività predefinita ($\alpha_{out}$), la procedura si arresta. Il modello corrente è il modello finale.
        - Altrimenti, si individua il predittore con il _p-value_ più alto e lo si rimuove dal modello.
    4. **Iterazione**: si torna al punto 1 con il nuovo modello e si ripete.

    In tal caso la soglia di significatività viene tipicamente scelta abbastanza alta $\alpha_{out}\approx 0.30$. Notare però che questo tipo di selezione risente fortemente della **multicollinearità**, ovvero la colrrelazioni tra le variabili d'ingresso, questo perché:
    - La multicollinearità influisce negativamente sulla stabilità numerica della matrice $X^TX$ aumentando la varianza degli stimatori OLS.
    - Ciò comporta:
      - Un'incremento degli errori standard.
      - Una riduzione delle statistiche $t$.
      - Un aumento dei _p-value_.

    Ciò rende alcune variabili erroneamente non significative nonostante il loro reale contributo al modello.


In [19]:
import pandas as pd
import statsmodels.api as sm

def backward_selection(x, y, alpha=0.002):    
    x_selected = x.copy()
    
    while True:
        model = sm.OLS(y, x_selected).fit()

        p_values = model.pvalues.drop("const", errors="ignore")
        
        max_p = p_values.max()
        
        if max_p > alpha:
            feature_to_remove = p_values.idxmax()
            x_selected = x_selected.drop(columns=[feature_to_remove])
        else:
            break
    
    return x_selected, model

dataframe = pd.read_csv("../db/health_fitness_dataset.csv")

y = dataframe["calories_burned"]
x = dataframe.drop(columns=["calories_burned", "participant_id", "date"])

x = pd.get_dummies(x, drop_first=True) # mette le variabili categoriche nella forma corretta
x = x.astype(float)
x = sm.add_constant(x)

x_selected, model = backward_selection(x, y)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        calories_burned   R-squared:                       0.889
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                 4.243e+05
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        18:50:46   Log-Likelihood:            -1.8020e+06
No. Observations:              687701   AIC:                         3.604e+06
Df Residuals:                  687687   BIC:                         3.604e+06
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

In tal caso i test sono avvenuti andando a diminuire man mano $\alpha$ fino cercando di mantenere i criteri di valutazione $R^2$ i più simili possibile.

- **Selezione forward**: si parte dal modello nullo contenente solo l'intercetta e si aggiungono man mano le variabili:
    1. **Inizio**: si parte dal modello nullo, contenente solo l'intercetta.
    2. **Aggiunta**: si provano ad aggiungere, una alla volta, tutte le variabili non ancora incluse nel modello. Si stima una regressione per ciascune di queste "prove".
    3. **Decisione**: si sceglie la variabile che, una volta aggiunta, migliora maggiormente il modello. Questa scelta viene guidata da indicatori globali:
        - La variabile che produce il modello con l'errore standard della regressione ($S_e$) più basso.
        - In modo equivalente, quella che produce l'$R^2_{\text{adj}}$ più alto.
    4. **Iterazione**: la variabile scelta viene aggiunta definitivamente al modello. Il ciclo riparte dal punto 2, provando ad aggiungere le restanti variabili al nuovo modello, finché non si raggiunge una condizione di arresto.

    La procedura si ferma quando l'aggiunta di una qualsiasi delle variabili rimanenti non porta a un miglioramento significativo del modello (ad esempio, quando l'$S_e$ smette di diminuire e inizia ad aumentare).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

def forward_selection(x, y):
    remaining_features = list(x.columns)
    selected_features = []
    
    current_se = np.inf 
    while remaining_features:
        results = []
        
        for candidate in remaining_features:
            features_to_test = selected_features + [candidate]
            x_test = sm.add_constant(x[features_to_test])
            
            model = sm.OLS(y, x_test).fit()
            se = np.sqrt(model.mse_resid)
            
            results.append((se, candidate, model))
        
        results.sort(key=lambda x: x[0])
        
        best_se, best_candidate, best_model = results[0]
        if best_se < current_se:
            remaining_features.remove(best_candidate)
            selected_features.append(best_candidate)
            
            current_se = best_se            
            final_model = best_model
        else:
            break
    
    x_selected = sm.add_constant(x[selected_features])
    return x_selected, final_model

y = dataframe["calories_burned"]
x = dataframe.drop(columns=["calories_burned", "participant_id", "date"])

x = pd.get_dummies(x, drop_first=True) # mette le variabili categoriche nella forma corretta
x = x.astype(float)
x = sm.add_constant(x)

x_selected, model = forward_selection(x, y)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        calories_burned   R-squared:                       0.889
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                 2.399e+05
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        18:45:04   Log-Likelihood:            -1.8019e+06
No. Observations:              687701   AIC:                         3.604e+06
Df Residuals:                  687677   BIC:                         3.604e+06
Df Model:                          23                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

Oltre ai metodi stepwise esistono dei **metodi globali o best subset selection** i quali adottano un approccio esaustivo, ovvero si testano tutti i possibili sottoinsiemi di variabili. Per $p$ predittori questo significa stimare e valutare $2^p$ modelli di regressione. Notare che a causa della crescita esponenziale del numero di modelli, questo approccio è fattibile solo per un numero di predittori non troppo grande. 

Nei metodi globali si utilizzeranno delle metriche di performance o punteggi (score) per valutare i modelli, i prncipali sono:
- $S_e$ (Errore Standard della Regressione): si cerca il modello con l'$S_e$ minimo.
- **Criteri basati sui coefficienti di determinazione**:
  - **Coefficiente di determinazione** ($R^2_D$): il coefficiente di determinazione, noto come $R^2$ o $R^2_D$, spiega quanto del fenomeno viene spiegato dal modello di regressione, il suo risultato sarà quindi una percentuale, ad esempio $R^2=0.85$ allora il modello spiega l'85% della variabilità delle calorie:
    $$R^2=1-\frac {\text{SSR}} {\text{SSY}}$$
    dove:
    -  SSY (Total Sum of Squares): è la devianza totale di $Y$, calcolata come $\sum_{i=1}^n (y_i - \overline y)^2$, ovvero la variabilità della variabile risposta prima di considerare i predittori.
    - SSR (Sum of Squared Residuals): è la devianza residua, calcolata come $\sum_{i=1}^n (y_i - \hat y_i)^2$, ovvero la parte di variabilità di $y$ che il modello non è riuscito a spiegare.
    però $R^2$ ha un grosso limite, ovvero aumenta sempre all'aumentare del numero di predittori $p$ del modello.

  - **Coefficiente di determinazione corretto** ($R^2_A$): supera il limite di $R^2_D$ andando a penalizzare l'aggiunta di variabili inutili:
    $$R^2_A = 1-\frac {S_e^2} {S_y^2}$$
    dove:
    - $S_e^2=\frac {\text{SSR}} {n-p-1}$ è la stima della varianza degli errori (MSE).
    - $S_y^2=\frac {\text{SSY}} {n-1}$ è la varianza campionaria di y.

- **AIC (Akaike Information Criterion)**: un criterio che bilancia la bontà di adattamente del modello con la sua complessità, la formula è:
    $$\text{AIC}=2k-2\ln(L)$$
    dove $k$ è il numero di parametri e $L$ è la verosomiglianza (Likelihood). Quindi si cercherà il modello con l'AIC minimo. Questa metrica ritorna utile perché tipicamente abbiamo che aumentando il numero di variabili diminuisce l'errore standard, perciò usando AIC andiamo a penalizzare per il numero di variabili.
- **Validazione**: si valutano le performance del modello (es. $S_e$, $SSR$) su un set di dati di validazione.

In [14]:
import pandas as pd
import statsmodels.api as sm
import itertools
import numpy as np

def best_subset_selection(x, y, p_max = 20, criterion="se"):
    features = list(x.columns)
    
    best_score = np.inf if criterion in ["se", "aic"] else -np.inf
    best_model = None
    best_subset = None
    
    all_results = []
    for k in range(1, min(p_max, len(features)) + 1):
        for subset in itertools.combinations(features, k):
            x_subset = sm.add_constant(x[list(subset)])
            model = sm.OLS(y, x_subset).fit()
            
            se = np.sqrt(model.mse_resid)
            aic = model.aic
            adj_r2 = model.rsquared_adj
            
            all_results.append({
                "subset": subset,
                "se": se,
                "aic": aic,
                "adj_r2": adj_r2
            })
            
            if criterion == "se":
                score = se
                if score < best_score:
                    best_score = score
                    best_model = model
                    best_subset = subset
                    
            elif criterion == "aic":
                score = aic
                if score < best_score:
                    best_score = score
                    best_model = model
                    best_subset = subset
                    
            elif criterion == "adj_r2":
                score = adj_r2
                if score > best_score:
                    best_score = score
                    best_model = model
                    best_subset = subset
    
    print(f"\nMiglior subset ({criterion}): {best_subset}")
    print(f"Score: {best_score}")
    
    return best_model, best_subset, pd.DataFrame(all_results)

dataframe = pd.read_csv("../db/health_fitness_dataset.csv")

y = dataframe["calories_burned"]

x = dataframe.drop(columns=["calories_burned", "participant_id", "date"])
x = pd.get_dummies(x, drop_first=True)
x = x.astype(float)

model, subset, results = best_subset_selection(x, y, p_max = 2, criterion="aic")

print(model.summary())


Miglior subset (aic): ('weight_kg', 'duration_minutes')
Score: 4574898.025273743
                            OLS Regression Results                            
Dep. Variable:        calories_burned   R-squared:                       0.545
Model:                            OLS   Adj. R-squared:                  0.545
Method:                 Least Squares   F-statistic:                 4.121e+05
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:39:24   Log-Likelihood:            -2.2874e+06
No. Observations:              687701   AIC:                         4.575e+06
Df Residuals:                  687698   BIC:                         4.575e+06
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------

Notare però che questo metodo se pur sembri ottimale diventa impraticabile quando il numero di parametri $p$ è grande, di fatti abbiamo che:
- $p=10$ avremo che il confronto sarà tra 1024 modelli.
- $p=20$ avremo che il confronto sarà tra circa 1 milione di modelli.

Perciò se di base guardando i dati sappiamo che determinate variabili possono risultare inutili allora è meglio fin da subito filtrare il numero di modelli. 

## Estensione dei metodi stepwise con analisi della Varianza (ANOVA)

L'analisi della Varianza (ANOVA) offre un metodo formale per confrontare modelli di regressione annidati, ovvero quando dati due insiemi di variabili candidate $C$ e $\tilde C$, tali che $C\subset \tilde C$, vogliamo testare se le variabili aggiuntive presenti in $\tilde C$ portano un contributo significativo al modello. Questo torna utile perché siccome il modello basato su $\tilde C$ contiene più variabili il suo adattamento ai dati sarà sempre migliore o uguale al modello basato su $C$, questo si traduce in:
$$
R^2_D(\tilde C) \ge R^2_D(C) \iff \text{SSR}(\tilde C) \le \text{SSR}(C)
$$
L'ANOVA ci permette di quantificare se la riduzione dell'errore (la differenza $\text{SSR}(C) - \text{SSR}(\tilde C)$) è statisticamente significativa o solo dovuta al caso. Per fare ciò si introduce la **Statistica F per il Confronto**, dove procedendo secondo l'inferenza statistica avremo che:
- $H_0: \beta_{j} = 0 \quad \forall j \in \tilde C / C$, quindi le nuove variabili non hanno nessuna relazione.
- $H_1: \beta_{j} \ne 0 \quad \forall j \in \tilde C / C$, quindi le nuove variabili hanno effettivamente una relazione. 

Per il calcolo della statistica test si definisce la somma dei quadrati addizionale ($\text{SS}_D$) come la riduzione dell'errore ottenuta passando dal modello più piccolo al modello più grande:
$$
\text{SS}_D = \text{SSR}(C) - \text{SSR}(\tilde C)
$$
e poi si calcola la statistica test per il confronto dei modelli come:
$$
V = \frac {\text{SS}_D / (\tilde d-d)} { \text{SSR}(C) / (n-d-1)}
$$
dove:
- $d$ è il numero di variabili del modello più piccolo $C$ 
- $\tilde d$ è il numero di variabili del modello più grande $\tilde C$
- $n$ è il numero di osservazioni.

Il _p-value_ si calcolerà come:
$$
\text{p-value} = 1- F_{(\tilde d - d, n-d-1)}(V)
$$

dove $F_{(\tilde d - d, n-d-1)}$ è la funzione di ripartizione della distribuzione F di Fisher la quale è dovuta al fatto che la statistica test $V$ viene costruita tramite il rapporto tra due entità che seguono distribuzione $\chi^2$ e il rapporto di due $\chi^2$ normalizzate segue una distribuzione $F$.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

def forward_selection_anova(X, y, alpha=0.05):
    remaining_features = list(X.columns)
    selected_features = []

    current_model = None

    while len(remaining_features) > 0:
        best_pvalue = np.inf
        best_feature = None
        best_model = None

        for candidate in remaining_features:
            features = selected_features + [candidate]
            X_candidate = sm.add_constant(X[features])

            model = sm.OLS(y, X_candidate).fit()

            if current_model is None:
                pvalue = model.f_pvalue
            else:
                X_current = sm.add_constant(X[selected_features])
                model_current = sm.OLS(y, X_current).fit()

                anova_result = sm.stats.anova_lm(model_current, model)
                pvalue = anova_result["Pr(>F)"][1]

            if pvalue < best_pvalue:
                best_pvalue = pvalue
                best_feature = candidate
                best_model = model

        if best_pvalue < alpha:
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            current_model = best_model
        else:
            break

    X_selected = sm.add_constant(X[selected_features])
    final_model = sm.OLS(y, X_selected).fit()

    return X_selected, final_model

dataframe = pd.read_csv("../db/health_fitness_dataset.csv")

y = dataframe["calories_burned"]

x = dataframe.drop(columns=["calories_burned", "participant_id", "date"])
x = pd.get_dummies(x, drop_first=True)
x = x.astype(float)
x = sm.add_constant(x)

x_selected, model = forward_selection_anova(x, y)

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        calories_burned   R-squared:                       0.889
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                 3.245e+05
Date:                Sat, 18 Apr 2026   Prob (F-statistic):               0.00
Time:                        21:50:01   Log-Likelihood:            -1.8019e+06
No. Observations:              687701   AIC:                         3.604e+06
Df Residuals:                  687683   BIC:                         3.604e+06
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
const         

## Regolarizzazione

La regolarizzazione è una tecnica utilizzata per evitare l'overfitting nei modelli di regressione, essa si basa sul fatto che:
> Molte volte quanto abbiamo un overfitting sui dati di traning accade che i coefficienti $\beta_j$ assumono valori molto grandi, questo accade perché il modello per ridurre l'errore sui dati di training tenta di forzare determinate relazioni, ciò porta a:
> - Piccoli cambiamenti negli input portano a grandi cambiamenti nell'output, quindi se i dati di test sono anche solo un pò diversi da quelli di training allora ci potrebbero essere risultati insensati.
> - La funzione diventa sensibile.

Allora per evitare questa problematica si aggiunge una penalità all'interno delle formula di minimizzazione dei quadrati residui (SSR), ovvero la formula che si va a minimizzare in modo nascosto nel momento in cui calcoliamo i coefficienti:
$$
\min_{\beta} (SSR) = \min_{\beta} (\sum_{i=1}^n (y_i - \sum_{j=0}^p \beta_j x_{ij})^2)
$$

Le principali tecniche di regolarizzazione sono:
- **Regressione Ridge (L2)**: la regressione Ridge aggiunge una penalità proporzionale alla somma dei quadrati dei coefficienti. L’obiettivo non è solo ridurre l’errore sui dati, ma anche mantenere i coefficienti piccoli.

    $$
    \min_{\beta} (\frac 1 n \sum_{i=1}^n (y_i - \sum_{j=0}^p \beta_j x_{ij})^2 + \alpha \sum_{j=0}^p \beta_j^2)
    $$

    dove:

    - $\alpha \geq 0$ è il parametro di regolarizzazione che controlla l'intensità della regolarizzazione.

    - maggiore è $\alpha$, più i coefficienti vengono “compressi” verso zero

    Questa regolarizzazione è efficiace nel ridurre la varianza del modello e tende a restringere i coefficienti verso lo zero senza però mai annullarli.

- **Regressione Lasso (L1)**: la regressione Lasso aggiunge invece una penalità basata sulla somma dei valori assoluti dei coefficienti.

    $$

    \min_{\beta} (\frac 1 n \sum_{i=1}^n (y_i - \sum_{j=0}^p \beta_j x_{ij})^2 + \lambda \sum_{j=0}^p |\beta_j| )

    $$

    La differenza principale con la regolarizzazione Ridge è la regolarizzazione Lasso può anche portare a 0 alcuni coefficienti, ciò quindi permette di effettuare una scelta automatica delle variabili.

Notare che nel momento in cui si sceglie di effettuare la regolarizzazione tipicamente non si utilizza la formula OLS per determinare i coefficienti, ma piuttosto si utilizzano metodi iterativi come nel Machine Learning normale.

Al giorno d'oggi per la scelta delle variabili si preferiscono utilizzare le tecniche di regolarizzazione, come Lasso o Elastic Net (combinazione tra L1 e L2) piuttosto che i metodi stepwise visti in precedenza.

### Normalizzare le variabili

Supponiamo due variabili:
- $x_1$: ore di allenamento ([0, 3]).
- $x_2$: calorie giornaliere [[0, 3000]].

per fare in modo che esse abbiano il medesimo effetto su $y$ OLS fa in modo che $x_1$ abbia un coefficiente grande e $x_2$ abbia un coefficiente piccolo, tuttavia come abbiamo appena visto la regolarizzazione va a penalizzare i coefficienti grandi, perciò è buona cosa normalizzare le variabili in modo che abbiano una scala standardizzata tra di loro, perciò tipicamente si trasforma ogni variabile in:
$$
x'=\frac {x - \mu} {\sigma}
$$

ciò fa in modo che:
- Media = 0.
- Deviazione standard = 1.

In [ ]:
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = dataframe["calories_burned"]
X = dataframe.drop(columns=["calories_burned", "participant_id", "date"])

X = pd.get_dummies(X, drop_first=True)
feature_names = X.columns

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = Lasso(alpha=0.1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

coefs = model.coef_

selected_features = feature_names[coefs != 0]

print("Variabili selezionate da Lasso:")
print(selected_features)

print("\nCoefficienti:")
for name, coef in zip(feature_names, coefs):
    if coef != 0:
        print(f"{name}: {coef}")

Variabili selezionate da Lasso:
Index(['weight_kg', 'duration_minutes', 'avg_heart_rate',
       'activity_type_Cycling', 'activity_type_Dancing', 'activity_type_HIIT',
       'activity_type_Running', 'activity_type_Swimming',
       'activity_type_Tennis', 'activity_type_Walking',
       'activity_type_Weight Training', 'activity_type_Yoga', 'intensity_Low',
       'intensity_Medium'],
      dtype='str')

Coefficienti:
weight_kg: 3.559950057620163
duration_minutes: 6.30471027756623
avg_heart_rate: 0.0872067534041069
activity_type_Cycling: 0.5110709466475156
activity_type_Dancing: -1.2739262129763425
activity_type_HIIT: 2.797786006696324
activity_type_Running: 1.3425731968990493
activity_type_Swimming: -0.28965194615194895
activity_type_Tennis: -0.09080466348746877
activity_type_Walking: -2.355458371146627
activity_type_Weight Training: -0.9464684431755548
activity_type_Yoga: -2.9171627694810223
intensity_Low: -2.019373010321613
intensity_Medium: -0.9712162066290329


In [ ]:
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = dataframe["calories_burned"]
X = dataframe.drop(columns=["calories_burned", "participant_id", "date"])

X = pd.get_dummies(X, drop_first=True)
feature_names = X.columns

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

model = ElasticNet(alpha=0.1, l1_ratio=0.5)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

coefs = model.coef_
selected_features = feature_names[coefs != 0]

print("Variabili selezionate da Elastic Net:")
print(selected_features)

print("\nCoefficienti:")
for name, coef in zip(feature_names, coefs):
    if coef != 0:
        print(f"{name}: {coef}")

Variabili selezionate da Lasso:
Index(['age', 'height_cm', 'weight_kg', 'duration_minutes', 'avg_heart_rate',
       'bmi', 'fitness_level', 'gender_M', 'activity_type_Cycling',
       'activity_type_Dancing', 'activity_type_HIIT', 'activity_type_Running',
       'activity_type_Swimming', 'activity_type_Tennis',
       'activity_type_Walking', 'activity_type_Weight Training',
       'activity_type_Yoga', 'intensity_Low', 'intensity_Medium'],
      dtype='str')

Coefficienti:
age: 0.13836052752038017
height_cm: 0.44619202473331226
weight_kg: 2.018263085441736
duration_minutes: 6.045466993116705
avg_heart_rate: 0.3585637348056395
bmi: 0.7048862034192847
fitness_level: 1.1425720000149635
gender_M: 0.012965967139241545
activity_type_Cycling: 0.5376035008930762
activity_type_Dancing: -1.2558782937524258
activity_type_HIIT: 2.7258993295240614
activity_type_Running: 1.333410819024593
activity_type_Swimming: -0.31445089988923075
activity_type_Tennis: -0.12466749431821049
activity_type_Walking: